# QuickStart: Your First NeuralForecast Model

## Learning Objectives

By completing this notebook you will:
1. Verify neuralforecast is installed and working
2. Load the French Bakery daily sales dataset
3. Train NHITS to produce a 7-day probabilistic forecast
4. Visualize the forecast with prediction intervals
5. Understand the output DataFrame structure

**Estimated time:** 12 minutes  
**Prerequisites:** Python 3.9+, neuralforecast installed (`pip install neuralforecast datasetsforecast utilsforecast`)

---

In [ ]:
learning_objectives([
    "12 minutes",
    "Python 3.9+, neuralforecast installed (`pip install neuralforecast datasetsforecast utilsforecast`)"
])

## Step 1: Verify Installation

Run this cell first. It confirms the three core libraries are available and prints their versions. If any import fails, run `pip install neuralforecast datasetsforecast utilsforecast` and restart the kernel.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
import neuralforecast
import datasetsforecast
import utilsforecast
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print(f"neuralforecast : {neuralforecast.__version__}")
print(f"datasetsforecast: {datasetsforecast.__version__}")
print(f"utilsforecast  : {utilsforecast.__version__}")
print(f"pandas         : {pd.__version__}")
print("\nAll imports OK — ready to forecast.")

## Step 2: Load the French Bakery Dataset

The French Bakery dataset contains daily sales figures for 8 bakery items. It lives in the nixtla transfer-learning repository and loads as a CSV. The result is already in the required three-column format:

- `unique_id` — item name (string series identifier)
- `ds` — date of the sale
- `y` — number of units sold

This format is the universal contract across all nixtla libraries.

In [ ]:
# Load French Bakery daily sales from the nixtla public repository
url = (
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series/"
    "main/datasets/french_bakery_daily.csv"
)
df = pd.read_csv(url, parse_dates=['ds'])

# Inspect the data
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nSeries in this dataset:")
print(df['unique_id'].unique())
print(f"\nDate range: {df['ds'].min().date()} to {df['ds'].max().date()}")
print(f"Rows per series: {df.groupby('unique_id').size().iloc[0]}")

## Step 3: Explore the Sales Patterns

Before fitting any model, look at the data. Bakery sales have strong weekly seasonality — weekends drive higher volumes for most items. Seeing this pattern now helps us interpret the model's lag importance later.

In [ ]:
# Plot daily sales for up to 4 items over the full date range
items = df['unique_id'].unique()[:4]

fig, axes = plt.subplots(len(items), 1, figsize=(14, 8), sharex=True)
if len(items) == 1:
    axes = [axes]

for ax, item in zip(axes, items):
    subset = df[df['unique_id'] == item].sort_values('ds')
    ax.plot(subset['ds'], subset['y'], linewidth=0.9, color='steelblue')
    ax.set_ylabel('Units sold', fontsize=9)
    ax.set_title(item, fontsize=10, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

axes[-1].set_xlabel('Date')
plt.suptitle('French Bakery: Daily Sales by Item', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## Step 4: Prepare the Train/Test Split

We hold out the last 7 days of each series for evaluation. The split must be applied per series using `groupby` + `apply` to ensure each series is split at the correct cutoff — not the global last 7 rows, which could mix items.

In [ ]:
HORIZON = 7  # forecast 7 days ahead

# Split each series independently at the horizon boundary
train = (
    df.groupby('unique_id', group_keys=False)
      .apply(lambda x: x.sort_values('ds').iloc[:-HORIZON])
      .reset_index(drop=True)
)

test = (
    df.groupby('unique_id', group_keys=False)
      .apply(lambda x: x.sort_values('ds').iloc[-HORIZON:])
      .reset_index(drop=True)
)

print(f"Train rows : {len(train):,}")
print(f"Test rows  : {len(test):,}  ({HORIZON} days × {test['unique_id'].nunique()} series)")
print(f"\nTrain cutoff per series:")
print(train.groupby('unique_id')['ds'].max())

## Step 5: Instantiate the Model

We configure NHITS with:
- `h=7` — 7-day forecast horizon (matches our test set)
- `input_size=28` — 28-day lookback window (4 weeks of history)
- `loss=MQLoss(...)` — distributional loss that outputs the 10th, 50th, and 90th percentiles
- `max_steps=500` — training iterations (sufficient for this small dataset)
- `scaler_type='standard'` — z-score normalization per series before training

We also include XLinear as a linear baseline. If NHITS does not outperform XLinear, the dataset is well-described by linear dynamics.

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, XLinear
from neuralforecast.losses.pytorch import MQLoss

QUANTILES = [0.1, 0.5, 0.9]  # 80% prediction interval centered on the median

nf = NeuralForecast(
    models=[
        # Linear baseline — fast, interpretable, always run first
        XLinear(h=HORIZON, input_size=4 * HORIZON),

        # NHITS with distributional loss — our primary model
        NHITS(
            h=HORIZON,
            input_size=4 * HORIZON,
            loss=MQLoss(quantiles=QUANTILES),
            max_steps=500,
            scaler_type='standard',
        ),
    ],
    freq='D',  # daily frequency
)

print("Models instantiated:")
for model in nf.models:
    print(f"  {model}")

## Step 6: Train the Models

`.fit()` trains all models in the `NeuralForecast` list simultaneously. A progress bar appears during training. On a standard laptop CPU, 500 steps on this dataset takes about 60–90 seconds.

In [ ]:
# Train on historical data — progress bar shows training iteration
nf.fit(df=train)
print("Training complete.")

## Step 7: Generate the 7-Day Forecast

`.predict()` generates out-of-sample forecasts starting the day after the last observed date in `train`. The output is a DataFrame with one row per (series, forecast day) and one column per model output.

In [ ]:
# Generate forecasts for the 7-day horizon
forecasts = nf.predict()

print("Forecast columns:")
print(forecasts.columns.tolist())
print()
print("Baguette forecast:")
print(forecasts[forecasts['unique_id'] == 'baguette'].to_string(index=False))

## Step 8: Visualize the Forecast

We overlay the point forecast (median quantile) and the 80% prediction interval on top of the observed history. The shaded region shows the range within which we expect 80% of actual future observations to fall — if the model is well-calibrated.

In [ ]:
def plot_probabilistic_forecast(train, test, forecasts, series_id, model_prefix='NHITS'):
    """Plot observed history, forecast median, prediction interval, and actual test values."""
    fig, ax = plt.subplots(figsize=(13, 5))

    # Historical data — last 60 observations for readability
    hist = train[train['unique_id'] == series_id].sort_values('ds').tail(60)
    ax.plot(hist['ds'], hist['y'],
            color='black', linewidth=1.2, label='Observed (train)', zorder=3)

    # Forecast median
    fc = forecasts[forecasts['unique_id'] == series_id].sort_values('ds')
    median_col = f'{model_prefix}-q-0.5'
    ax.plot(fc['ds'], fc[median_col],
            color='steelblue', linewidth=2, linestyle='--',
            label='Median forecast', zorder=4)

    # 80% prediction interval (10th to 90th percentile)
    ax.fill_between(
        fc['ds'],
        fc[f'{model_prefix}-q-0.1'],
        fc[f'{model_prefix}-q-0.9'],
        alpha=0.25, color='steelblue', label='80% prediction interval',
    )

    # Actual test values
    actual = test[test['unique_id'] == series_id].sort_values('ds')
    ax.scatter(actual['ds'], actual['y'],
               color='crimson', zorder=5, s=50, label='Actual (test)')

    # Vertical line at train/test boundary
    cutoff = hist['ds'].max()
    ax.axvline(cutoff, color='grey', linewidth=1, linestyle=':', alpha=0.8)

    ax.set_title(f'{series_id} — NHITS 7-Day Forecast with 80% Interval')
    ax.set_xlabel('Date')
    ax.set_ylabel('Daily sales')
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()


# Plot baguette — the highest-volume item
plot_probabilistic_forecast(train, test, forecasts, series_id='baguette')

## Step 9: Plot All Series

The same visualization applied to every item in the dataset. This gives a quick visual calibration check: do the red dots (actual values) fall inside the blue shaded region roughly 80% of the time?

In [ ]:
# Panel plot: one subplot per bakery item
series_ids = df['unique_id'].unique()
n_series = len(series_ids)
ncols = 2
nrows = (n_series + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows), sharey=False)
axes_flat = axes.flat

for ax, series_id in zip(axes_flat, series_ids):
    hist = train[train['unique_id'] == series_id].sort_values('ds').tail(30)
    fc = forecasts[forecasts['unique_id'] == series_id].sort_values('ds')
    actual = test[test['unique_id'] == series_id].sort_values('ds')

    ax.plot(hist['ds'], hist['y'], color='black', linewidth=0.9)
    ax.plot(fc['ds'], fc['NHITS-q-0.5'],
            color='steelblue', linewidth=1.8, linestyle='--')
    ax.fill_between(
        fc['ds'], fc['NHITS-q-0.1'], fc['NHITS-q-0.9'],
        alpha=0.25, color='steelblue',
    )
    ax.scatter(actual['ds'], actual['y'], color='crimson', s=35, zorder=5)
    ax.set_title(series_id, fontsize=9)
    ax.set_ylabel('Units', fontsize=8)

# Hide any unused subplots
for ax in list(axes_flat)[n_series:]:
    ax.set_visible(False)

plt.suptitle('NHITS 7-Day Forecasts — All Bakery Items', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## Step 10: Inspect the Output Structure

Understanding the forecast DataFrame structure is essential before proceeding to evaluation and downstream use. Every column follows the pattern `{ModelName}-q-{quantile}` for probabilistic outputs, or `{ModelName}` for point outputs.

In [ ]:
print("Forecast DataFrame info:")
print(f"  Shape: {forecasts.shape}")
print(f"  Rows = {df['unique_id'].nunique()} series × {HORIZON} horizon steps")
print()
print("Column breakdown:")
for col in forecasts.columns:
    if col in ('unique_id', 'ds'):
        print(f"  {col:<25} → index column")
    elif col == 'XLinear':
        print(f"  {col:<25} → XLinear point forecast (MAE loss)")
    else:
        parts = col.split('-q-')
        print(f"  {col:<25} → {parts[0]} model, {float(parts[1]):.0%} quantile")

print()
print("Range of forecast values (baguette):")
bag = forecasts[forecasts['unique_id'] == 'baguette']
print(bag[['NHITS-q-0.1', 'NHITS-q-0.5', 'NHITS-q-0.9']].describe().round(1))

In [ ]:
section_divider("Summary")

## Summary

You have completed the NeuralForecast quickstart.

### What you did
1. Loaded the French Bakery daily sales dataset in the nixtla `(unique_id, ds, y)` format
2. Split each series into train and test sets at a 7-day horizon
3. Instantiated NHITS with `MQLoss` for probabilistic output and XLinear as a baseline
4. Trained both models with `.fit(df=train)`
5. Generated 7-day forecasts with `.predict()`
6. Visualized the forecast with shaded 80% prediction intervals

### Key patterns to remember
- `NeuralForecast(models=[...], freq='D')` wraps multiple models in one API
- `MQLoss(quantiles=[...])` turns any model into a probabilistic forecaster
- Forecast columns follow the pattern `{ModelName}-q-{quantile}`
- The split must be done per series using `groupby` + `apply`

### What's next
- **Notebook 02:** Explore three datasets and learn EDA patterns for time series
- **Guide 02:** Deep dive on the neuralforecast API including `.cross_validation()`, `.simulate()`, and `.explain()`
- **Module 01:** Point forecasting — NHITS and NBEATS architectures, benchmark comparisons

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])